# Figure 2 panels F & G -- abundance heat-trees (virus families, nematode genera)

> **Public release copy.** Set `<DATA_DIR>` in the CONFIG cell to the folder holding the input
> taxonomy tables (see **Inputs** below). Runs in R.
>
> Standalone notebook for **Figure 2 panels F & G**. The virus input is the same per-contig taxonomy table the Figure 4 curation
> notebook starts from; the nematode input is the analogous table for nematodes. Both are direct
> outputs of the metatranscriptomic pipeline.

Two metacoder heat-trees, each with node size/color = cumulative relative abundance:

- **Panel F** -- viruses, collapsed to **family** level (top 200 families by abundance):
  `taxonomy_hits_viruses_abundance_200families`
- **Panel G** -- nematodes, collapsed to **genus** level (top 600 genera by abundance):
  `taxonomy_hits_nematodes_abundance_genus`

Both trees are built the same way (the work is factored into one helper), so the only
differences are the input table and the taxonomic rank they collapse to.

## Inputs (set `<DATA_DIR>`; not included here)

- **Virus taxonomy:** the per-contig virus taxonomy table from the metatranscriptomic
  pipeline (the same `taxonomy_hits_viruses_clusters_notargetsnosequenceforsalmon_*` table used
  by the Fig 4 curation notebook).
- **Nematode taxonomy:** the per-contig nematode taxonomy table (`taxonomy_hits_nematoda_*`),
  also a direct output of the same metatranscriptomic pipeline.

Both carry the standard `tax_*_NTorNR` rank columns and a `rel_abundance` column.

## Outputs

- `taxonomy_hits_viruses_abundance_200families_<date>.pdf` / `.png`  (Panel F)
- `taxonomy_hits_nematodes_abundance_genus_<date>.pdf` / `.png`      (Panel G)


## CONFIG -- edit paths here

In [ ]:
library(tidyverse)
library(metacoder)

# -- CONFIG -- edit paths here -----------------------------------------------
workingpath <- getwd()

virus_taxonomy_file    <- "<DATA_DIR>/taxonomy_hits_viruses_clusters_notargetsnosequenceforsalmon_mostrecent.tsv"
nematode_taxonomy_file <- "<DATA_DIR>/taxonomy_hits_nematoda.tsv"

today <- Sys.Date()
# ----------------------------------------------------------------------------

## Helper: build one abundance heat-tree

Given a per-contig taxonomy table, this:
1. drops rows with `clade`/`group` (and a few known-problematic clades) in the lineage,
2. drops rows with no value at the target rank,
3. keeps the top-N taxa at the target rank by total `rel_abundance`,
4. builds a `;`-joined classification string up to the target rank,
5. parses a metacoder taxmap, computes per-taxon abundances, and
6. draws the heat-tree (node size/color = cumulative abundance, fixed `c(1e1,1e7)` scale,
   `set.seed(11)` for a reproducible davidson-harel layout).

In [ ]:
# rank_cols: lineage columns to include in the classification, up to and INCLUDING the target rank
abundance_heattree <- function(data, target_rank_col, rank_cols, top_n, out_basename) {
  cat("\n=== building heat-tree:", out_basename, "===\n")

  # 1. drop problematic taxonomy (whole-word 'clade'/'group'; a few known bad clades)
  lineage_cols <- c("tax_superkingdom_NTorNR","tax_clade_NTorNR","tax_kingdom_NTorNR",
                    "tax_phylum_NTorNR","tax_class_NTorNR","tax_order_NTorNR","tax_family_NTorNR")
  lineage_cols <- intersect(lineage_cols, names(data))
  clean <- data %>%
    filter(!if_any(all_of(lineage_cols),
                   ~ !is.na(.x) & str_detect(as.character(.x),
                       regex("\\bclade\\b|\\bgroup\\b", ignore_case = TRUE)))) %>%
    filter(is.na(tax_clade_NTorNR) | !str_detect(tax_clade_NTorNR,
        "Norovirus GI|Norovirus GII|Human|recombinant Vesiculovirus|papillomaviruses|Grandeviruses|amphibian retroviruses|recombinants"))

  # 2. drop rows with no value at the target rank
  clean <- clean %>% filter(!is.na(.data[[target_rank_col]]))
  n_taxa <- n_distinct(clean[[target_rank_col]])
  cat("  rows:", nrow(clean), "| distinct", target_rank_col, ":", n_taxa, "\n")

  # 3. keep top-N target-rank taxa by total rel_abundance
  if (n_taxa > top_n) {
    keep <- clean %>% group_by(.data[[target_rank_col]]) %>%
      summarise(tot = sum(rel_abundance, na.rm = TRUE), .groups = "drop") %>%
      arrange(desc(tot)) %>% head(top_n) %>% pull(1)
    clean <- clean %>% filter(.data[[target_rank_col]] %in% keep)
    cat("  kept top", top_n, "by abundance ->", nrow(clean), "rows\n")
  }

  # 4. classification string up to the target rank
  temp <- clean %>% rowwise() %>%
    mutate(classification = paste(na.omit(unlist(across(all_of(rank_cols)))), collapse = ";")) %>%
    ungroup()
  uniq <- temp %>% distinct(classification) %>% filter(classification != "")
  cat("  unique classifications:", nrow(uniq), "\n")

  # 5. parse taxmap + per-taxon abundances
  obj <- parse_tax_data(uniq, class_cols = "classification", class_sep = ";")
  obj$data$query_data <- temp %>%
    left_join(obj$data$tax_data %>% select(taxon_id, classification), by = "classification") %>%
    filter(!is.na(taxon_id))
  num_cols <- names(obj$data$query_data)[sapply(obj$data$query_data, is.numeric)]
  obj$data$tax_abund <- calc_taxon_abund(obj, "query_data", cols = num_cols)
  cat("  taxa:", length(obj$taxon_ids()), "\n")

  # 6. heat-tree (reproducible layout)
  set.seed(11)
  ht <- heat_tree(obj,
    node_size  = obj$data$tax_abund$rel_abundance,
    node_color = obj$data$tax_abund$rel_abundance,
    node_label = taxon_names,
    node_label_size_range = c(0.005, 0.015),
    node_label_max = 800, tree_label_max = 800,
    node_color_digits = 0, node_size_digits = 0,
    repel_force = 1.5,
    node_size_interval  = c(1e1, 1e7),
    node_color_interval = c(1e1, 1e7),
    node_size_axis_label = "Cumulative Abundance",
    node_color_axis_label = "Cumulative Abundance",
    layout = "davidson-harel", initial_layout = "reingold-tilford")

  ggsave(paste0(out_basename, "_", today, ".pdf"), ht, width = 28, height = 25, units = "in", limitsize = FALSE)
  ggsave(paste0(out_basename, "_", today, ".png"), ht, width = 28, height = 25, units = "in", limitsize = FALSE)
  cat("  wrote", out_basename, "\n")
  ht
}

## Panel F -- viruses to family (top 200 families)

In [ ]:
virus_data <- read_tsv(virus_taxonomy_file, show_col_types = FALSE) %>%
  dplyr::filter(viruscategorysimple_NTorNR != "Adapter")

family_rank_cols <- c("tax_superkingdom_NTorNR","tax_clade_NTorNR","tax_kingdom_NTorNR",
                      "tax_phylum_NTorNR","tax_class_NTorNR","tax_order_NTorNR","tax_family_NTorNR")

# -> Figure 2F
heattree_virus_family <- abundance_heattree(
  virus_data, target_rank_col = "tax_family_NTorNR",
  rank_cols = family_rank_cols, top_n = 200,
  out_basename = "taxonomy_hits_viruses_abundance_200families")

## Panel G -- nematodes to genus (top 600 genera)

In [ ]:
nematode_data <- read_tsv(nematode_taxonomy_file, show_col_types = FALSE) %>%
  arrange(desc(rel_abundance)) %>%
  mutate(taxname_lca_NTorNR = str_replace_all(taxname_lca_NTorNR, "[()]", " "))

genus_rank_cols <- c(family_rank_cols, "tax_genus_NTorNR")

# -> Figure 2G
heattree_nematode_genus <- abundance_heattree(
  nematode_data, target_rank_col = "tax_genus_NTorNR",
  rank_cols = genus_rank_cols, top_n = 600,
  out_basename = "taxonomy_hits_nematodes_abundance_genus")